# Lesson 58 · Watching the machine learn — the five-step closed loop of the training loop, fitting the two moons

**Goal**: get the 5-step training loop actually running: `forward → loss → backward → update → zero_grad`, train an MLP with your home-made autograd to fit the two moons, and plot the loss curve.

> **The five-step checklist, up front**: `forward → loss → backward → update → zero_grad`. Chain these five steps into a closed loop and you can read the training code of every deep-learning framework.

🔗 **Aurora connection**: the Month 2 PyTorch training scripts in `src/aurora/` have exactly the same structure as this lesson's loop — the five steps `forward → loss → backward → update → zero_grad` are word for word the same in every deep-learning framework; understand this loop and you understand all deep-learning training.

← **Previous lesson** · [L57 · An MLP from scratch](L57_mlp.ipynb)

> Last lesson covered **an MLP from scratch**: hand-written fully connected layers, activations, and a complete forward / backward implementation.
> This lesson turns **the training loop** into a training script that really converges.

## Parts check: `Value` + `Neuron` + loss + optimizer

- `Value` provides `backward()` and gradient accumulation
- `Neuron` / `Layer` handle the forward mapping
- the loss function squeezes the classification objective into a single scalar
- `optimizer.step()` / `zero_grad()` close one round of training

Line these four parts up in your head first, then read the 5-step training loop below.

## This lesson's story: why does a neural network need a "5-step loop"?

You already have an `MLP` that can run a forward pass. But to make it "learn" there's one step missing: **how do we move the parameters in the direction that reduces the loss?**

The answer is **gradient descent**: the partial derivative of the loss function with respect to each parameter tells it "which way to go, and how far". The complete loop has just 5 steps:

```
for epoch in range(N):
    1. ypred = [model(x) for x in xs]          # forward: compute the predictions
    2. loss = hinge_loss(ypred, ys)             # compute the loss (a scalar)
    3. loss.backward()                           # backward: fill in every parameter's .grad
    4. for p in model.parameters(): p.data -= lr * p.grad   # update the parameters
    5. for p in model.parameters(): p.grad = 0.0            # zero the gradients
```

**Why step 5 can't be skipped**: `Value._backward()` accumulates gradients with `+=`. Without zeroing, the next round's gradient piles on top of what's left from the previous round, the parameter updates grow larger and larger, and the loss diverges.

**Hinge loss (SVM style)**: labels `y ∈ {+1, -1}`, score `s = model(x).data`.
```
loss_i = max(0, 1 - y_i * s_i)   → if y_i * s_i ≥ 1 (correct and confident), loss=0
                                    → if y_i * s_i < 1 (wrong or not confident), loss>0
```

Your task in this lesson: implement `train(model, xs, ys, lr, epochs)` — the 5-step loop + a record of the loss history.

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

# Assumes the Value / MLP defined in L57_mlp.ipynb have been run in the same session
# If you run this notebook on its own, paste them in or %run L57_mlp.ipynb first
# Below is a minimal version of the Value + MLP definitions again. It differs from L57 on purpose in two ways:
# ① the hidden-layer activation switches back from L57's tanh to relu — the original micrograd approach,
#    and this lesson's hinge-loss demo needs only one nonlinear operator, relu (gradients propagate the same way for both)
# ② the output-layer activation problem is fixed: the output layer has nonlin=False, so output scores can be negative

class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self): return f'Value({self.data:.4f})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out
    def __radd__(self, other): return self + other

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward
        return out
    def __rmul__(self, other): return self * other

    def __neg__(self): return self * -1
    def __sub__(self, other): return self + (-other)
    def __rsub__(self, other): return Value(other) - self
    def __truediv__(self, other): return self * Value(other)**-1 if not isinstance(other, Value) else self * other**-1

    def __pow__(self, exp):
        assert isinstance(exp, (int, float))
        out = Value(self.data**exp, (self,), f'**{exp}')
        def _backward():
            self.grad += exp * (self.data**(exp-1)) * out.grad
        out._backward = _backward
        return out

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')
        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out

    def backward(self):
        topo, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for c in v._prev: build(c)
                topo.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(topo): v._backward()


class Neuron:
    def __init__(self, n_in, nonlin=True):
        self.w = [Value(random.uniform(-1,1)) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin        # the output layer passes nonlin=False: no ReLU applied
    def __call__(self, x):
        act = sum(wi*xi for wi,xi in zip(self.w,x)) + self.b
        return act.relu() if self.nonlin else act   # the output layer returns the linear activation directly
    def parameters(self): return self.w + [self.b]

class Layer:
    def __init__(self, n_in, n_out, nonlin=True):
        self.neurons = [Neuron(n_in, nonlin=nonlin) for _ in range(n_out)]
    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs)==1 else outs
    def parameters(self): return [p for n in self.neurons for p in n.parameters()]

class MLP:
    def __init__(self, n_in, layer_sizes):
        sizes = [n_in] + layer_sizes
        # last layer nonlin=False: output scores can be negative, so hinge loss can converge properly
        # consistent with the original karpathy/micrograd implementation
        self.layers = [Layer(sizes[i], sizes[i+1],
                             nonlin=(i < len(layer_sizes)-1))
                       for i in range(len(layer_sizes))]
    def __call__(self, x):
        for layer in self.layers: x = layer(x)
        return x
    def parameters(self): return [p for l in self.layers for p in l.parameters()]

print('Value + MLP defined ✅  (hidden layers use relu, output layer activation switched off with nonlin=False)')


## 1. The structure of the training loop

The standard training loop has five steps, and none of them can be left out:

```
for epoch in range(N):
    ypred = [model(x) for x in xs]   # forward
    loss  = compute_loss(ypred, ys)   # loss
    loss.backward()                   # backward
    for p in model.parameters():      # update
        p.data -= lr * p.grad
    for p in model.parameters():      # zero_grad
        p.grad = 0.0
```

What each step does: **forward** produces the predictions; **loss** squeezes "how bad is it" into a single scalar; **backward** propagates that scalar's gradient back to every parameter; **update** nudges each parameter against the gradient; **zero_grad** clears this round's gradients so they don't accumulate into the next round.

In [ ]:
# Demo: the simplest possible one-step training loop
random.seed(0)
tiny_model = MLP(2, [1])        # 2-D input, 1 scalar output
x_demo = [Value(0.5), Value(-0.3)]
y_demo = 1.0                     # target class +1

score = tiny_model(x_demo)
hinge = (Value(1.0) - Value(y_demo) * score).relu()   # max(0, 1 - y*score)
print(f'score={score.data:.4f}  hinge={hinge.data:.4f}')
hinge.backward()
print('parameter gradients (first 5):', [f'{p.grad:.4f}' for p in tiny_model.parameters()[:5]])
# take one update step
lr = 0.05
for p in tiny_model.parameters(): p.data -= lr * p.grad
for p in tiny_model.parameters(): p.grad = 0.0
print('one update step done ✅')

## 2. Hinge loss (an SVM-style loss)

For binary classification, the labels are `y ∈ {+1, -1}` and the network outputs one real-valued `score`.

The hinge loss is defined as:
```
L_i = max(0,  1 - y_i * score_i)
```

When the prediction points the right way with enough confidence (`y*score >= 1`), the loss is 0; otherwise the loss grows linearly.

Mean loss over the batch:
```
L = mean(L_i  for  i in range(N))
```

Compared with cross-entropy, hinge loss is sparser (most correctly classified samples have zero gradient), which suits SVMs and contrastive learning. We pick it here because it uses only addition and `relu`, entirely within what our home-made `Value` supports.

In [ ]:
# Visualize how hinge loss changes with y*score
margin = np.linspace(-1.5, 2.5, 200)
hinge_vals = np.maximum(0, 1 - margin)
plt.figure(figsize=(5,3))
plt.plot(margin, hinge_vals, lw=2, color='steelblue')
plt.axvline(1, color='gray', ls='--', lw=1, label='margin=1')
plt.xlabel('y × score')
plt.ylabel('hinge loss')
plt.title('Hinge Loss = max(0, 1 - y·score)')
plt.legend()
plt.tight_layout()
plt.show()

### Why does hinge loss use `y × score`? A trick that unifies the two labels

Classification usually uses `y ∈ {0, 1}`, but here we use `y ∈ {+1, -1}`. The key is **using multiplication to unify the target**.

Imagine two samples:
- Sample A: y=+1 (positive class), we want score > 1
- Sample B: y=-1 (negative class), we want score < -1

If you wrote a separate target formula for score in each case, you'd need two different if-else lines. But with `y × score`:
- Sample A: y×score = (+1)×(2.5) = +2.5  ✓ positive → loss=0
- Sample B: y×score = (-1)×(-1.5) = +1.5  ✓ also becomes positive → loss=0

**In one sentence**: the beauty of y ∈ {+1, -1} is that **multiplication can flip the target**, so both the positive and the negative class use the same loss formula.

### Where does margin=1 come from?

The hinge loss `max(0, 1 - y·score)` can be rewritten as `max(0, margin - y·score)`, where `margin=1` is the **safety margin of the classification**:
- if you're sure a sample is correct (y·score ≥ 1), the loss is 0 and the model can relax
- if y·score hasn't reached 1 yet (say 0.8), then even though the prediction points the right way, the model keeps pushing until it reaches the boundary

**Why 1 and not 0.5 or 2?** It's a **conventional unit** — margin=1 is the standard in the SVM literature (it's tied to feature scaling). In practice you can tune it, but 1 is the most common starting point.

### Why are sparse gradients a good thing?

Compare how the two losses' gradients behave:
- **Cross-entropy**: even when the prediction is correct, a small gradient keeps fine-tuning the parameters
- **Hinge loss**: once y·score ≥ 1, the gradient becomes 0 and the model stops adjusting

The benefits of sparse gradients:
1. **Efficient computation**: many correct samples produce no gradient, saving computation
2. **Overfitting control**: samples that are already correct no longer adjust the parameters, preventing overfitting
3. **Friendly to contrastive learning**: well suited to learning relative distances between samples (as in face recognition)

### The gradient of hinge loss: deriving a piecewise function

Hinge loss is a `relu` function of `y·score`:
```
loss = relu(1 - y·score) = max(0, 1 - y·score)
```

Differentiate (chain rule):
```
∂loss / ∂score = ∂relu(·) / ∂(1-y·score)  ×  ∂(1-y·score) / ∂score
                = [1-y·score > 0 ? 1 : 0]  ×  (-y)
                = {  -y           if y·score < 1
                  {  0            if y·score ≥ 1
```

**Concrete examples**:
- positive sample (y=+1) with score=0.5 (y·score=0.5 < 1): gradient = -1, the parameters should increase the score
- positive sample (y=+1) with score=1.5 (y·score=1.5 ≥ 1): gradient = 0, the model can relax, no adjustment
- negative sample (y=-1) with score=-2.0 (y·score=+2.0 ≥ 1): gradient = 0, the model can relax
- negative sample (y=-1) with score=0.5 (y·score=-0.5 < 1): gradient = +1, the parameters should decrease the score

## 3. zero_grad: why must we zero every round?

In the implementation of `Value.backward()`, gradients are **accumulated** into `.grad` (`self.grad += ...`). This is so that when a node is used along several paths, the gradients add up correctly.

But across epochs, `.grad` does not reset to zero by itself. If you don't zero it by hand:
```
epoch1: p.grad = 0.3
epoch2: p.grad = 0.3 + 0.3 = 0.6   # wrong!
epoch3: p.grad = 0.9               # bigger and bigger, the parameter updates blow up
```

So before each round's `backward` (or after `update`) you must run:
```python
for p in model.parameters():
    p.grad = 0.0
```
In PyTorch this corresponds to `optimizer.zero_grad()`, and the principle is exactly the same.

### At which step of the loop should zero_grad run?

The standard five-step loop runs in this order:

```python
for epoch in range(epochs):
    ypred = [model(x) for x in xs]      # step 1: forward
    loss = compute_loss(ypred, ys)       # step 2: compute the loss
    loss.backward()                      # step 3: backprop fills in the gradients
    for p in model.parameters():         # step 4: update the parameters (against the gradient)
        p.data -= lr * p.grad
    for p in model.parameters():         # step 5: zero the gradients
        p.grad = 0.0
```

**The key rule**: zero_grad must happen **after the parameter update and before the next round's backward**. In this lesson's layout, the natural place for it is **the end of the loop** (step 5).

### What about zeroing just before backward?

If you move zero_grad in front of backward:
```python
for p in model.parameters(): p.grad = 0.0   # ← zero
loss.backward()                              # ← fill in the gradients
for p in model.parameters(): p.data -= lr * p.grad  # update with the fresh gradients
```

This works just as well — in fact it's the order you'll often see in PyTorch code (`optimizer.zero_grad()` right before `loss.backward()`). Either position is fine, as long as the zeroing falls between one round's update and the next round's backward.

### Why aren't the gradients duplicated in the first epoch?

Students often ask: "Before the first epoch the parameters' `.grad` is 0.0, so after the first backward's `self.grad += ...` it's just a single gradient, right? So there's no need to zero?"

**The answer**: in the first epoch the initial gradient is 0 so you can't see any accumulation, but **from the second epoch on, it starts piling up**. If you forget to zero:
```
epoch 1: p.grad = 0 + grad1 = grad1
epoch 2: p.grad = grad1 + grad2 = grad1 + grad2  (accumulated! not just grad2)
epoch 3: p.grad = (grad1 + grad2) + grad3        (still accumulating)
```

So zeroing is a must, so that **each round sees only the current epoch's gradient**.

In [ ]:
# Demo: the harm done by gradient accumulation
random.seed(42)
m = MLP(2, [1])
x = [Value(1.0), Value(0.5)]
y = 1.0

print('how .grad changes without zeroing:')
for i in range(3):
    s = m(x)
    loss = (Value(1.0) - Value(y) * s).relu()
    loss.backward()   # no zeroing
    grad_0 = m.parameters()[0].grad
    print(f'  epoch {i+1}: params[0].grad = {grad_0:.4f}')

print('\nhow .grad changes with zeroing:')
for p in m.parameters(): p.grad = 0.0   # zero once first
for i in range(3):
    s = m(x)
    loss = (Value(1.0) - Value(y) * s).relu()
    loss.backward()
    grad_0 = m.parameters()[0].grad
    print(f'  epoch {i+1}: params[0].grad = {grad_0:.4f}')
    for p in m.parameters(): p.grad = 0.0   # zero every round

## 4. ✏️ Implement `train(model, xs, ys, lr=0.05, epochs=100)`

**A 5-step implementation template**:

| Step | Action | Common mistake |
|---|---|---|
| 1 | `ypred = [model(x) for x in xs]` | forgetting to redo the forward pass every round (reusing an old ypred) |
| 2 | `loss = sum((Value(1) - y*yp).relu() ...)` | using plain Python floats everywhere, which breaks the gradient chain |
| 3 | `loss.backward()` | forgetting to set loss.grad=1 first (backward() already handles this internally) |
| 4 | `p.data -= lr * p.grad` | use `-=` to modify `.data` directly; don't touch `.grad` |
| 5 | `for p in params: p.grad = 0.0` | skipping the zeroing, so the next round's gradients pile up |

**Signature**:
```python
def train(model, xs, ys, lr=0.05, epochs=100) -> list[float]:
    """Return a list of float loss values, one per epoch (loss_history)"""
```

**Acceptance criteria**:
- returns a list of length `epochs`
- `loss_history[-1] < loss_history[0]` (the loss goes down)
- after 100 epochs, accuracy on the moons dataset > 85% (with the random seed fixed, the reference implementation gets about 95%)

### Core details: learning rate, parameter updates, computation graph

Before filling in `train()`, get a grip on these concepts — they're often treated as "obvious", but beginners easily get stuck on them.

#### ① What is the learning rate? Why multiply by `lr`?

The gradient tells you the **direction**, but not the **step size**. Think of walking down a staircase:
- the gradient ∇loss is like a "downward arrow" — the direction is right
- the learning rate lr is like "how far each step goes down" — how big is the step?

If `∇w = -0.3` (a negative gradient: the parameter should increase), you have three choices:
```
p.data -= 0.001 * (-0.3)    # tiny step: slow progress, but stable
p.data -= 0.05 * (-0.3)     # medium step: balances progress and stability
p.data -= 0.5 * (-0.3)      # huge step: may overshoot the optimum → oscillate and diverge
```

**The mathematical form**: the parameter update rule is called **gradient descent**
```
θ_new = θ_old - lr × ∇loss(θ_old)
```

That's why the code says `p.data -= lr * p.grad`.

#### ② Why must the output layer have `nonlin=False`?

In the hinge loss formula `max(0, 1 - y*score)`, **the score must be able to go negative**. If the output layer has a ReLU:

```python
# Wrong (activation on the output layer):
output_layer = Layer(..., nonlin=True)  # ReLU: score >= 0
```

then for a negative sample (y=-1), even if the network wants to learn score=-2 (meaning "very sure this is the negative class"), the ReLU clamps it to 0, so that:
- negative samples can't be represented correctly (score is always ≥ 0)
- the hinge loss formula breaks down (y*score ≥ 1 can never be reached)
- training gets stuck and the loss can't go down

**The right way** (already done in the code):
```python
# output layer activation switched off
self.layers = [
    Layer(2, 16, nonlin=True),     # hidden layer 1: ReLU
    Layer(16, 16, nonlin=True),    # hidden layer 2: ReLU
    Layer(16, 1, nonlin=False),    # output layer: linear activation (pass-through)
]
```

**The general rule**: hinge loss / SVMs need a linear output; cross-entropy can use softmax; squared loss can use a linear or another activation. Choose to match what the loss function needs.

#### ③ Batch training: why run forward on every sample?

The code looks like this:
```python
ypred = [model(x) for x in xs]
```

Compared with running forward on just one sample, a batch has several advantages:
- **More stable gradients**: a single sample's gradient can be very noisy (its direction jitters); a batch gradient averages many samples → smoother
- **Computational efficiency**: modern GPUs are highly optimized for batched operations; processing 32 samples at once is much faster than processing 1 sample 32 times
- **Better generalization**: the randomness of mini-batch gradients naturally acts as "regularization", guarding against overfitting

In this exercise we run forward on all 200 moon-shaped samples at once (equivalent to batch_size=200), which is called **full-batch training**.

#### ④ Mean loss and the chain rule: why does averaging make the gradient 1/N?

Suppose there are N samples, each with loss `L_i = max(0, 1 - y_i*s_i)`.

The mean loss:
```
loss = (L_1 + L_2 + ... + L_N) / N
```

During backpropagation (chain rule):
```
∂loss / ∂s_i = ∂[(L_1+...+L_N)/N] / ∂s_i
             = (1/N) × ∂L_i / ∂s_i
```

**Intuitively**: the partial derivative of loss with respect to L_i is 1/N (the averaging coefficient), which is then multiplied by L_i's partial derivative with respect to the parameters. This 1/N propagates through the chain rule into every parameter's gradient.

In the code this is handled automatically — you only compute the mean loss `loss = sum(...) / N` and call `loss.backward()`, and the chain rule automatically hands that 1/N factor to every sample.

#### ⑤ Value objects and the computation graph: why not use Python floats?

A simple example: compute the derivative of `z = (x + 1) * 2` with respect to `x`.

**With Python floats** (information is lost):
```python
x = 3.0
z = (x + 1) * 2  # z = 8.0, but the information about the intermediate steps is lost
# dz/dx = ? impossible to trace back, because we don't know how z was computed from x
```

**With Value objects** (the computation graph is recorded):
```python
x = Value(3.0)
y = x + 1         # y remembers: "I = x + 1, my parent is x"
z = y * 2         # z remembers: "I = y * 2, my parent is y"
z.backward()      # trace the computation graph back from z
                  # ∂z/∂y = 2 (chain)
                  # ∂y/∂x = 1 (derivative of addition)
                  # ∂z/∂x = ∂z/∂y × ∂y/∂x = 2 × 1 = 2
```

**The essence of the computation graph**: each Value object stores its **source operation** and its **parent nodes**. `backward()` simply **walks this graph in reverse**, going back from the output and applying the chain rule layer by layer.

**Why accumulate gradients with `+=`?** If a node is used along several paths, its gradient should be the **sum** of the contributions from all paths. For example:
```python
x = Value(2.0)
z = (x + x) * 2   # x is used twice
z.backward()      # x.grad should be 2 (one path) + 2 (the other path) = 4
```

In [ ]:
def train(model, xs, ys, lr=0.05, epochs=100):
    loss_history = []
    for epoch in range(epochs):
        # ✏️ TODO: forward — compute predictions for all samples
        ypred = None  # replace this line

        # ✏️ TODO: compute each sample's hinge loss = max(0, 1 - y*pred)
        losses = None  # replace this line

        # ✏️ TODO: take the mean to get the scalar loss
        loss = None   # replace this line

        # ✏️ TODO: backpropagate

        # ✏️ TODO: update the parameters (p.data -= lr * p.grad)

        # ✏️ TODO: zero the gradients (p.grad = 0.0)

        # stub guard: give a friendly hint while the TODO is unfinished, instead of an AttributeError
        if loss is None:
            raise NotImplementedError(
                'TODO not finished. Implement, in order:\n'
                '  ypred  = [model(x) for x in xs]\n'
                '  losses = [(Value(1.0) - Value(y)*p).relu() for p,y in zip(ypred,ys)]\n'
                '  loss   = sum(losses) * (1.0/len(losses))\n'
                '  loss.backward()\n'
                '  for p in model.parameters(): p.data -= lr * p.grad\n'
                '  for p in model.parameters(): p.grad = 0.0'
            )

        loss_history.append(loss.data)
        if (epoch + 1) % 20 == 0:
            print(f'epoch {epoch+1:3d}  loss={loss.data:.4f}')
    return loss_history


In [ ]:
# ── Generate the moons dataset ──
try:
    from sklearn.datasets import make_moons
except ImportError:
    # if sklearn isn't installed, implement make_moons from scratch in numpy with the same interface (two interleaved half-moon arcs + Gaussian noise)
    # the point distribution matches sklearn's; with random_state fixed the result is reproducible
    def make_moons(n_samples=100, noise=None, random_state=None):
        rng = np.random.default_rng(random_state)
        n_out = n_samples // 2          # upper arc (class 0)
        n_in = n_samples - n_out        # lower arc (class 1)
        t_out = np.linspace(0, np.pi, n_out)
        t_in = np.linspace(0, np.pi, n_in)
        X = np.concatenate([
            np.stack([np.cos(t_out), np.sin(t_out)], axis=1),
            np.stack([1 - np.cos(t_in), 1 - np.sin(t_in) - 0.5], axis=1),
        ])
        y = np.concatenate([np.zeros(n_out, dtype=int), np.ones(n_in, dtype=int)])
        if noise is not None:
            X = X + rng.normal(scale=noise, size=X.shape)
        return X, y
    print("sklearn not installed — using the from-scratch make_moons (same interface, reproducible)")
X_raw, Y_raw = make_moons(n_samples=200, noise=0.15, random_state=7)
# convert labels to ±1
Ys = [2*y-1 for y in Y_raw.tolist()]
# convert to lists of Values
Xs = [[Value(x) for x in row] for row in X_raw.tolist()]

# initialize the MLP: 2-D input, hidden layers 16+16, 1 scalar output
random.seed(0)
model = MLP(2, [16, 16, 1])
print(f'total parameters: {len(model.parameters())}')

# ── Train ──
history = train(model, Xs, Ys, lr=0.05, epochs=100)

# ── Acceptance check ──
with_pred = [model([Value(x) for x in row]).data for row in X_raw.tolist()]
acc = sum(1 for p,y in zip(with_pred, Ys) if (p>=0) == (y==1)) / len(Ys)
print(f'\nfinal accuracy: {acc*100:.1f}%')
assert history[-1] < history[0], 'the loss should go down'
assert acc > 0.85, f'accuracy {acc:.2f} is not above 85%'
print('✅ training acceptance check passed')

# ── Plot the loss curve ──
plt.figure(figsize=(5,3))
plt.plot(history, lw=2, color='steelblue')
plt.xlabel('epoch')
plt.ylabel('hinge loss')
plt.title('Training loss curve (lr=0.05)')
plt.tight_layout()
plt.show()

# ── Visual check: the decision boundary ──
grid_step = 0.05
xx, yy = np.meshgrid(np.arange(X_raw[:,0].min()-0.3, X_raw[:,0].max()+0.3, grid_step),
                      np.arange(X_raw[:,1].min()-0.3, X_raw[:,1].max()+0.3, grid_step))
Z = np.array([model([Value(float(a)), Value(float(b))]).data
              for a,b in zip(xx.ravel(), yy.ravel())]).reshape(xx.shape)
plt.figure(figsize=(5,4))
plt.contourf(xx, yy, Z, levels=0, alpha=0.3, colors=['#f77', '#77f'])
plt.scatter(X_raw[:,0], X_raw[:,1], c=['red' if y==1 else 'blue' for y in Ys], s=15)
plt.title('Decision boundary (visual check)')
plt.tight_layout()
plt.show()

## 5. Parameter experiment: how the learning rate affects convergence

Train for 150 epochs each with `lr=0.001`, `lr=0.05` and `lr=0.5`, and plot the three loss curves:
- `lr=0.001`: the step is too small; the loss falls extremely slowly and still hasn't converged after 150 epochs
- `lr=0.05`: converges steadily; the loss falls smoothly
- `lr=0.5`: the step is too large; the loss oscillates wildly and may diverge

In [ ]:
EPOCHS = 150
results = {}
for lr_val in [0.001, 0.05, 0.5]:
    random.seed(0)
    m = MLP(2, [16, 16, 1])
    print(f'\n── lr={lr_val} ──')
    hist = train(m, Xs, Ys, lr=lr_val, epochs=EPOCHS)
    results[lr_val] = hist

plt.figure(figsize=(6,4))
colors = {0.001:'orange', 0.05:'steelblue', 0.5:'crimson'}
for lr_val, hist in results.items():
    plt.plot(hist, label=f'lr={lr_val}', color=colors[lr_val], lw=1.8)
plt.xlabel('epoch')
plt.ylabel('hinge loss')
plt.title('How the learning rate affects convergence speed / stability')
plt.legend()
plt.ylim(0, max(results[0.5][:10])*1.1 if max(results[0.5][:5])>1 else 1.5)
plt.tight_layout()
plt.show()

### Heuristics for choosing a learning rate

The plot above shows three extremes:

| Learning rate | What you see | Diagnosis |
|---|---|---|
| **Too small** (0.001) | the loss falls extremely slowly and still hasn't converged after 150 epochs | the parameter updates are too small; computation is wasted |
| **Just right** (0.05) | the loss falls steadily and gets close to the optimum within a few dozen epochs | the standard, stable and efficient setting |
| **Too large** (0.5) | the loss oscillates, or even diverges | the parameters "overshoot" the optimum and swing back and forth |

**How do you pick a suitable learning rate?**

1. **Rules of thumb** (a quick start):
   - start from `lr=0.01` or `lr=0.05`
   - if the loss falls too slowly (still no clear improvement after a few dozen epochs), raise lr
   - if the loss oscillates or diverges, lower lr

2. **Watch the loss curve**:
   - a smooth, monotonic decline → lr is fine
   - a slow decline → raise lr (try ×10)
   - oscillating up and down → lower lr (try ÷10)
   - diverging fast → lr is too large; lower it at once

3. **Advanced methods** (beyond this lesson):
   - learning rate scheduler: adjust lr dynamically during training
   - learning-rate warm-up: use a small lr for the first few epochs, then increase it gradually
   - adaptive optimizers (Adam/RMSprop): adjust each parameter's lr automatically

For this exercise's moons dataset, **lr=0.05 is a safe choice** — the loss falls smoothly and reaches high accuracy in around 100 epochs. In practice, the advice is to **try 0.01 first and adjust if it doesn't work**.

## Wrap-up

The `train()` function strings together the complete gradient-descent (GD) training loop: `forward → hinge loss → backward → update → zero_grad`, and returns the list of loss values, one per epoch. After 100 epochs the accuracy on the moons dataset is above 85%, showing that this closed loop can already push the parameters steadily in the right direction. This loop has exactly the same structure as the Month 2 PyTorch version in `src/aurora/` — the parameter-update logic, the timing of zero_grad and the way the loss is aggregated are word for word the same. The next lesson (L59) steps into the world of PyTorch: starting from `torch.Tensor`, you'll get hold of tensor operations, dtype conversion and device management, laying the groundwork for building with `nn.Module` later.

## ✏️ Whiteboard challenge: work out the training loop by hand (target: 10 minutes)

Cover the screen and derive on paper:

**Question 1**: Why must gradient descent zero `.grad` every round? Does the zeroing go before `backward()` or after `step()`?
(backward uses +=; without zeroing, gradients keep piling up round after round; the zeroing must come after the update and before the next round's backward)

**Question 2**: When is the gradient of hinge loss `max(0, 1-y*score)` zero?
(When y*score ≥ 1, loss=0 and that sample produces no gradient — the model is already correct enough)

**Question 3**: `lr=0.001` vs `lr=0.5`: which is more likely to diverge? Which converges slowly?
(lr=0.5 takes steps that are too large, so the parameters may overshoot the optimum → oscillate and diverge; lr=0.001 takes steps that are too small → slow convergence)

**Question 4**: In the training loop, what's the difference between `for x in xs: model(x)` and `model(xs)`?
(xs is a list; calling one at a time builds an independent computation graph for each sample; a batched call needs a model that supports batch input)

**Question 5**: In `train()`, loss is a `Value` object. Why should `loss_history` store `loss.data` (a float) rather than `loss` (a Value)?
(Storing the Value object keeps a reference to the whole computation graph, so the memory is never freed; storing the .data float keeps only the number)

When you've finished the derivations, run the cell below to check.

In [ ]:
# ✏️ Answer-check cell
import random

# Q1: zeroing gradients (verify the harm of accumulation)
try:
    random.seed(42)
    m_chk = MLP(2, [1])
    x_chk = [Value(1.0), Value(0.5)]
    out1 = m_chk(x_chk); out1.backward()
    grad_after_1 = [p.grad for p in m_chk.parameters()]
    out2 = m_chk(x_chk); out2.backward()
    grad_after_2 = [p.grad for p in m_chk.parameters()]
    # without zeroing, the second gradient is 2× the first (accumulated)
    doubled = all(abs(g2 - 2*g1) < 1e-10 for g1, g2 in zip(grad_after_1, grad_after_2))
    assert doubled, "gradients did not accumulate as expected"
    print(f"Q1 ✅  without zeroing, grad after the 2nd backward ≈ 2× the 1st (accumulated) — verified")
except (NotImplementedError, TypeError):
    print(f"Q1 ✅  why gradients accumulate: backward uses +=, so without zeroing they pile up every round (to be verified once implemented)")

# Q2: when the hinge-loss gradient is 0
# (score, y, nonzero gradient?): y*score ≥ 1 → loss=0, no gradient; y*score < 1 → loss>0, gradient
margin_vals = [(-0.5, -1, True), (0.5, 1, True), (1.0, 1, False), (1.5, 1, False)]
for score, y, expect_nonzero in margin_vals:
    yscore = y * score
    loss_val = max(0.0, 1 - yscore)
    grad_is_zero = (loss_val == 0.0)
    assert grad_is_zero == (not expect_nonzero), f"score={score},y={y}"
print(f"Q2 ✅  Hinge loss: loss=0 (zero gradient) when y*score≥1, loss>0 (nonzero gradient) when y*score<1")

# Q3: lr and convergence (concept check, no full training run)
lr_advice = {0.001: "converges slowly", 0.05: "just right", 0.5: "may diverge"}
for lr_v, advice in lr_advice.items():
    print(f"Q3 ✅  lr={lr_v}: {advice}")

# Q4/Q5: returns floats, not Values
try:
    random.seed(0)
    m_tiny = MLP(2, [1])
    xs_tiny = [[Value(1.0), Value(0.0)], [Value(0.0), Value(1.0)]]
    ys_tiny = [1.0, -1.0]
    hist = train(m_tiny, xs_tiny, ys_tiny, lr=0.05, epochs=5)
    assert isinstance(hist[0], float), f"loss_history should store floats, got {type(hist[0])}"
    assert len(hist) == 5, f"should return 5 values, got {len(hist)}"
    print(f"Q4/Q5 ✅  train() returns a list of floats, length={len(hist)}, first value={hist[0]:.4f}")
except (NotImplementedError, TypeError):
    print(f"Q4/Q5 ✅  loss_history stores floats (.data), not Value objects, to avoid a memory leak (to be verified once implemented)")
print("\n🎉 Training-loop whiteboard challenge passed!")

In [ ]:
# ✏️ Self-review for this lesson
l58_review = {
    "5step_loop_understood":   None,  # can you write the 5-step training loop from memory (forward/loss/backward/step/zero_grad)? True/False
    "train_implemented":       None,  # train() implemented correctly, loss goes down, accuracy>85%? True/False
    "zero_grad_reason":        None,  # understand why not zeroing makes gradients accumulate → divergence? True/False
    "hinge_loss_understood":   None,  # understand that hinge loss has zero gradient when y*score≥1? True/False
    "whiteboard_passed":       None,  # all 5 whiteboard derivations done? True/False
}

unfilled = [k for k, v in l58_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l58_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L58 fully cleared! On to L59: PyTorch tensors')

---

→ **Next lesson** · [L59 · PyTorch tensors](L59_tensor_basics.ipynb)

> Next lesson covers **PyTorch tensors**: converting to and from NumPy, devices, and `requires_grad`.